# Integrated Intelligent Systems: Tutorial 1 - Foundations (3 Pts)

This notebook introduces the fundamental components, challenges, and definitions, central to designing and implementing Cognition-Enabled Robot Agents.

## Part 1: The Unified Theoretical Exercise (E1–E6) (1.5 Pt)

**Title:** From Natural Language Request to action execution: The Eva Coffee Scenario

![Alternative Text](coffee-task-photo.png)

### Scenario Background
Eva requests: "Bring me my coffee." Her usual blue mug is in the dishwasher; a white mug is available on the counter. The fill level can vary, and Eva may shift position during execution.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)
Students work through this unified exercise in the VRB or on paper, structured across the six developmental steps of Lecture 1:

1. **E1 · Opening Judgment (Data vs. Competence):**
   - If an end-to-end vision-language-action model or a demonstration video shows a robot successfully serving coffee 100 times in a row, does that prove the robot "can do the task" across any home environment? State your reasoning and rate your confidence from 1 to 5.
   
     No, that demonstration video only shows that the robot successfully performed that task under the tested conditions. The robot might still fail serving cofee in other home environments, or when faced with an unexpected event. In order to claim the robot is capable of performing that task, we would need to test a much wider variety of situations and have information about the robot's failures. Confidence 5/5

2. **E2 · Changed Case Prediction (Physical Bounds):**
   - A half-full white mug was carried successfully along the route. Suppose the mug is now nearly full. Predict the physical consequence during a turn, and identify the untested assumption in the original run.
  
     Since the mug is now nearly full, it is much more likely that the coffee inside will spill over during the turn. Just because we were sucessful in carrying the half-full mug does not mean we have the guarantee that the same result will be reproduced with a full mug.
     

3. **E3 · The Commitment Decision (Warrant Gate):**
   - The blue mug is unavailable and white-mug acceptance is unconfirmed. Choose whether the robot should ask, observe, or proceed. State the single item of evidence that justifies your choice.
  
     The robot should definitely ask Eva whether the white mug is acceptable. The robot is able to detect the white mug but it hasn't established whether that mug is acceptable or not. It needs to ground its current action in the situation, which in this case requires Eva to specify if the white mug complies with her request.
     

4. **E4 · The Five Questions of Understanding (T1–T5):**
   - Formulate one distinct question for each dimension—Explain ($\text{T1}$), Predict ($\text{T2}$), Design ($\text{T3}$), Bound ($\text{T4}$), and Check ($\text{T5}$)—concerning the coffee delivery episode. At least one question must address a changed condition that a video alone cannot answer.

    The following are questions I devised for each dimension:
    - Explain (why did something happen?): why did the coffee not spill during the original delivery?
    - Predict (what would happen in a different situation?):  would the coffee spill if the robot made the same turn with a full mug?
    - Design (can we improve the solution?): would a tray or a new gripper make the coffee delivery more reliable?
    - Bound (under which conditions does the method work?): does the method work for all types of mugs and fill levels?
    - Check (what evidence supports our conclusion?): do we have records of the robot's actions, observations and outcomes that demonstrate the delivery was successful?
     
    The question I've come up with for the Predict dimension addresses a changed condition that a video can't answer.

5. **E5 · System Decomposition (Mapping to Architecture):**
   - List the core functional modules required to handle Eva's request (interpreting, perceiving, proposing, predicting, warranting, acting, and recording) and map them onto the Integration Diagram.

    In this question, we will map the actions required for delivering Eva's coffee to the Integration Diagram>
   - Interpret and Ground: understanding which drink and mug Eva wants
   - Perceive: detect the mug, potential obstacles and Eva's location
   - Know and Predict: estimate if a proposed series of movements would result in the robot colliding with objects or spilling the coffee
   - Propose: generate possible carry actions
   - Decision: Warrant Gate: determine whether an action meets the required criteria/ if the available evidence justify the action
   - Act and Regulate: execute and adjust movements
   - Record and Justify: store decisions, observations and the result
     

6. **E6 · Transfer & Exit Ticket (Justified Adaptation):**
   - Suppose Eva stands up while the robot is carrying the coffee. Revise the action, predict the potential failure mechanism, and state the exact evidence criterion required before the robot commits to the revised path.


    The robot should first stop and then use its sensors to determine Eva's new position. Afterwards, it should come up with a new route to get to her, as continuing along the same trajectory can result in the robot colliding with her. Before commiting to that new plan, it should be able to justify its actions, therefore it should have evidence that following that series of actions won't result in the coffee spilling over and colliding with an object or with Eva (we could consider that there needs to be at least half a meter separating the robot from any obstacle at all times as it is moving). Finally, the robot should be able to monitor the actions it is performing and be ready to halt them should any problem arise.
    

## Part 2: Interactive Virtual Lab & Python Tutorial (1.5 Pt)

**Title:** Implementing the Embodied Cognitive Architecture

### Scenario Background
Eva requests: "Bring me my coffee."

---

In [90]:
#!/usr/bin/env python3
"""Embodied Cognitive Architecture
"""

from dataclasses import dataclass, field
from typing import Dict, Any, Tuple, Optional, List
import random
import time

@dataclass(frozen=True)
class GoalSpecification:
    """Formalized, concrete goal specification derived via semantic interpretation."""
    target_object: str
    destination: str
    target_volume_ml: float
    max_allowable_force_n: float
    max_slosh_wave_mm: float
    max_nav_velocity_mps: float
    time_frame_seconds: float

@dataclass
class ParameterPacket:
    """System 1 (The Proposer): Flexible sampler proposing execution parameters."""
    
    #For p2 we need to know how deeply the robot presses the pouch, the proposed action needs to have that information; we'll give it a default value so that the rest of the code does not break
    grasp_pose: Tuple[float, float, float]
    max_velocity_mps: float
    tilt_rate_radps: float
    source_generator: str = "stochastic_sampler"
    contact_depth_m: float = 0.0
    
    



#At this stage the robot knows what Eva wants and what the environment looks like from its belief state, so it needs to decide how to proceed -> propose actions
#The proposed movement parameters are stored in a ParamterPacket object -> 3 params: grasp_pose, max_velocity_mps, tilt_rate_radps

class Proposer:
    #We'll use a uniform distribution, meaning every value in a specific interval has the same chance of being selected
    def __init__(self):
        pass

    def propose(self, speed=None):
        #We'll generate random values for the speed and tilt rate, but we'll keep the grasp pose constant; we'll have an option specify the speed
        if speed is None:
            speed = random.uniform(0.1, 0.5)
        tilt_rate = random.uniform(0.01, 0.10)
    
        proposal = ParameterPacket(grasp_pose=(0.45, 0.12, 0.30), max_velocity_mps = speed, tilt_rate_radps = tilt_rate)

        return proposal

@dataclass
class Observation:
    """Raw, noisy sensor measurements at time t (distinct from internal belief)."""
    timestamp: float
    measured_temperature_c: float
    measured_friction: float
    eva_position: str
    room_busyness: str

#we need to a battery level to the belief state, default value = full battery
@dataclass
class BeliefState:
    """The robot's current internal belief state (updated via perception)."""
    pouch_present: bool = True
    pouch_material: str = "flexible_plastic"
    fluid_viscosity_pas: float = 1.2
    vial_ready: bool = True
    camera_active: bool = True
    fallback_path_exists: bool = True
    audit_logger_enabled: bool = True
    # Real-time environmental parameters maintained internally
    ambient_temperature_c: float = 21.5
    surface_friction_coefficient: float = 0.45
    eva_position: str = "living_room"
    room_busyness: str = "medium"  # 'low', 'medium', 'high'
    preferred_cup: str = "blue_ceramic_mug"
    
    battery_level_percent: float = 100.0


#The robot needs to transform vague instructions into specific instructions it can use

class InterpretationSystem:
    """Uses a semantic knowledge base to map vague instructions and context 
    into precise physical goal parameters (GoalSpecification).
    """
    def __init__(self, knowledge_base: Dict[str, Any]):
        self.kb = knowledge_base


    #Produces a GoalSpecification object that stores the specific instructions (that were converted from the vague instructions)
    def interpret(self, raw_utterance: str, belief: BeliefState) -> GoalSpecification:
        print(f"\n[INTERPRETATION] Analyzing instruction: '{raw_utterance}' in context of Eva (Position: {belief.eva_position}, Room Busyness: {belief.room_busyness})")

        #Read what type of mug the robot CURRENTLY believes Eva prefers from its belief state
        cup_type = belief.preferred_cup
        cup_properties = self.kb["cups"].get(cup_type, {"max_force": 5.0, "max_wave": 2.0})
        
        busyness_factor = self.kb["room_busyness_limits"].get(belief.room_busyness, 0.5)
        max_nav_vel = 1.2 * busyness_factor

        #If Eva's instruction contains the word "fast" then the time_frame to accomplish the task is set to 30s, else 60s
        is_urgent = "schnell" in raw_utterance.lower() or "fast" in raw_utterance.lower()
        time_frame = 30.0 if is_urgent else 60.0
        target_volume = 15.0 if not is_urgent else 12.0

        #Store this information in a GoalSPecification object, which we return
        goal = GoalSpecification(
            target_object=cup_type,
            destination=belief.eva_position,
            target_volume_ml=target_volume,
            max_allowable_force_n=cup_properties["max_force"],
            max_slosh_wave_mm=cup_properties["max_wave"],
            max_nav_velocity_mps=max_nav_vel,
            time_frame_seconds=time_frame
        )
        
        print(f" -> Derived Goal: Object={goal.target_object}, MaxForce={goal.max_allowable_force_n}N, MaxNavVel={goal.max_nav_velocity_mps}m/s, TimeFrame={goal.time_frame_seconds}s")
        return goal

#Collects observations/ simulates the robot's sensors

class SensingSystem:
    """Simulates real-time noisy hardware sensors emitting raw observations."""
    def __init__(self):
        pass
        
    #Provides 4 pieces of information about the current environment:
    #1 The temperature of the room
    #2 Friction of the surface the robot moves on
    #3 Eva's position
    #4 How busy the room is? 
    
    def sense(self, true_ambient_temp: float, true_friction: float, eva_pos: str, busyness: str) -> Observation:
        
        #Sensors aren't perfect so the temperature/friction they measure might not correspond to the real one = simulate an error
        noisy_temp = true_ambient_temp + random.uniform(-0.5, 0.5)
        noisy_friction = true_friction + random.uniform(-0.05, 0.05)

        #Creates an Observation object containing the measures of the sensors, that can be passed to another system to be used
        return Observation(
            timestamp=time.time(),
            measured_temperature_c=round(noisy_temp, 2),
            measured_friction=round(noisy_friction, 2),
            eva_position=eva_pos,
            room_busyness=busyness
        )

#Uses observations to update the robot's understanding of the world
    
class PerceptionSystem:
    """Estimator module that updates the internal BeliefState using raw Observations."""
    def __init__(self, belief_state: BeliefState):
        self.belief_state = belief_state

    #The belief state is whatever information the robot currently believes to be true about its environment
    #Update the robot's belief state using the readings from the Observation object, i.e., the sensor's readings
    
    def update(self, observation: Observation) -> None:
        self.belief_state.ambient_temperature_c = observation.measured_temperature_c
        self.belief_state.surface_friction_coefficient = observation.measured_friction
        self.belief_state.eva_position = observation.eva_position
        self.belief_state.room_busyness = observation.room_busyness


class SemanticDigitalTwin:
    """System 2 (Semantic Digital Twin): Imperfect models with sim-to-real epistemic gaps."""
    def __init__(self, belief_state: BeliefState):
        self.belief = belief_state

    def predict_properties(self, params: ParameterPacket, goal: GoalSpecification) -> Dict[str, float]:
        #params is the ParameterPacket object so we need to check its contact depth
        stiffness_coefficient = 12.5 * (1.0 + self.belief.surface_friction_coefficient * 0.1)

        #We suppose the stiffness increases exponentially fy a factor of 1.8 EVERY cm if the contact depth is beyond 2cm
        #new_stiffness = old_stiffness * 1.8^^n where n is the number of cm the contact depth goes OVER 2cm

        if params.contact_depth_m > 0.02:
            excess_depth = params.contact_depth_m - 0.02
            n = excess_depth/0.01
    
            stiffness_coefficient = stiffness_coefficient * (1.8**n)

        
        #force = stiffness * movement speed ^^ 1.5 -> if speed constant constant and speed doubles = force doubles
        predicted_force = stiffness_coefficient * (params.max_velocity_mps ** 1.5)

        
        #Viscosity decreases by 1% for each deagree that the temperature increases over 20 degrees
        #1 -> calculate temperature difference compared to 20 degrees
        #2 -> calculate viscosity decrease: 1% * temp difference
        #3 -> new_viscosity = old_viscosity * (1 - viscosity_decrease)
        effective_viscosity = self.belief.fluid_viscosity_pas * (1.0 - (self.belief.ambient_temperature_c - 20.0) * 0.01)
    
        radius = 0.05

        #higher viscosity -> lower predicted wave height and vice-versa
        #temp increases -> viscosity decreases -> more liquid movement
        wave_height_m = (params.tilt_rate_radps * radius) / effective_viscosity

        #*1000 to convert to mm
        predicted_wave = wave_height_m * 1000.0

        # Injecting minor epistemic simulation error (Sim-to-Real gap)
        epistemic_force_error = random.uniform(-0.2, 0.3)
        epistemic_wave_error = random.uniform(-0.1, 0.2)

        return {
            "predicted_force_n": round(max(0.0, predicted_force + epistemic_force_error), 2),
            "predicted_wave_mm": round(max(0.0, predicted_wave + epistemic_wave_error), 2),
            "predicted_nav_velocity": min(params.max_velocity_mps, goal.max_nav_velocity_mps)
        }



#The Proposer class proposes an action, but before executing that action, the robot needs to justify if the action is acceptable
#The semantic digital twin will predict the consequences of executing an action, and the warrant gate will compare those predictions to the task requirements and verify if executing it is acceptable
    
class WarrantGate:
    """Validates predicted properties against the interpreted GoalSpecification."""
    def __init__(self, sdt: SemanticDigitalTwin):
        self.sdt = sdt

    
    def validate(self, params: ParameterPacket, goal: GoalSpecification, belief: BeliefState) -> Dict[str, Any]:

        #We ask the semantic digital twin to predict the consequences of an action
        #params = proposed action (movement)
        #goal = actual task instructions (not vague)
        #predictions = stores the result
        
        predictions = self.sdt.predict_properties(params, goal)

        #we need to define an estimated_battery_drain parameter which we'll base on the robot's speed -> higher speed higher consumption
        estimated_battery_drain = params.max_velocity_mps * 40.0
        
        #We check 7 requirements + 1 part 4
        checks = {
            "grounded": belief.vial_ready and belief.pouch_present,
            "feasible": params.max_velocity_mps > 0.0 and params.tilt_rate_radps > 0.0,
            "safe": (predictions["predicted_force_n"] <= goal.max_allowable_force_n) and 
                    (predictions["predicted_wave_mm"] <= goal.max_slosh_wave_mm) and
                    (predictions["predicted_nav_velocity"] <= goal.max_nav_velocity_mps),
            "monitorable": belief.camera_active,
            "repairable": belief.fallback_path_exists,
            "recordable": belief.audit_logger_enabled,
            #additional requirement for part 4, needs to be lower than 15%"
            #"energetically_viable": estimated_battery_drain < 15.0
            "energetically_viable": (estimated_battery_drain < 15.0 and belief.battery_level_percent >= estimated_battery_drain)
        }

        #If they all pass = all() returns true
        all_passed = all(checks.values())

        #We can check the first requirement that failed
        failed_criterion = next((name for name, passed in checks.items() if not passed), None)
        
        return {
            "passed": all_passed,
            "failed_criterion": failed_criterion,
            "predictions": predictions,
            "warrant_checks": checks
        }


#Executes an action and records what happens

class ExecutionEngine:
    """Manages physical execution and constructs the epistemic audit trail."""
    def __init__(self):
        self.audit_trail: List[Dict[str, Any]] = []

    
    def execute_action(self, params: ParameterPacket, goal: GoalSpecification, warrant_result: Dict[str, Any]) -> Dict[str, Any]:
        print(f"\n[ACTUATOR] Executing motion in physical world: v_max={params.max_velocity_mps} m/s")
        time.sleep(0.1)
        
        # Real-world divergence check
        predictions = warrant_result["predictions"]

        #The result of a prediction might be different from the result of physically performing that action = simulate error -> predictions not guarantees
        actual_force = predictions["predicted_force_n"] + random.uniform(-0.3, 0.6)
        actual_wave = predictions["predicted_wave_mm"] + random.uniform(-0.2, 0.5)

        #There's a spill if the force applied goes over a threshold, or if the liquid mvnt also goes over a limit
        spill_occurred = (actual_force > goal.max_allowable_force_n) or (actual_wave > goal.max_slosh_wave_mm)
        status = "FAILURE" if spill_occurred else "SUCCESS"
        
        execution_outcome = {
            "actual_force_n": round(actual_force, 2),
            "actual_wave_mm": round(actual_wave, 2),
            "actual_volume_transferred_ml": goal.target_volume_ml if not spill_occurred else goal.target_volume_ml - 3.0,
            "spill_occurred": spill_occurred,
            "status": status
        }
        return execution_outcome


    #The audit_trail list will record the robot's decisions and actions
    def record_trace(self, asked_utterance: str, goal: GoalSpecification, belief: BeliefState, 
                     proposal: ParameterPacket, warrant_result: Dict[str, Any], 
                     observation: Observation, committed: bool, execution_outcome: Optional[Dict[str, Any]] = None) -> None:
        
        #Dictionary containing information about the episode
        record = {
            "timestamp": time.time(),
            "asked": {
                "raw_utterance": asked_utterance,
                "interpreter_output": {
                    "target": goal.target_object,
                    "max_force_limit_n": goal.max_allowable_force_n,
                    "max_nav_velocity": goal.max_nav_velocity_mps
                }
            },
            "belief": {
                "eva_position": belief.eva_position,
                "room_busyness": belief.room_busyness,
                "ambient_temp_c": belief.ambient_temperature_c,
                "surface_friction": belief.surface_friction_coefficient
            },
            "observed": {
                "timestamp": observation.timestamp,
                "measured_temp_c": observation.measured_temperature_c,
                "measured_friction": observation.measured_friction,
                "eva_position": observation.eva_position,
                "room_busyness": observation.room_busyness
            },
            "committed": {
                "action_approved": committed,
                "proposed_parameters": {
                    "max_velocity_mps": proposal.max_velocity_mps,
                    "tilt_rate_radps": proposal.tilt_rate_radps
                },
                "warrant_verdict": warrant_result
            },
            "execution_outcome": execution_outcome
        }
        self.audit_trail.append(record)


# --- Execution Demonstration ---
if __name__ == "__main__":
    knowledge_base = {
        "cups": {
            "blue_ceramic_mug": {"max_force": 4.5, "max_wave": 1.5},
            "plastic_cup": {"max_force": 7.0, "max_wave": 3.0}
        },
        "room_busyness_limits": {
            "low": 1.0,
            "medium": 0.6,
            "high": 0.3
        }
    }

    # 1. Initialize systems
    belief_state = BeliefState(room_busyness="medium", preferred_cup="blue_ceramic_mug")
    sensing = SensingSystem()
    perception = PerceptionSystem(belief_state)
    interpreter = InterpretationSystem(knowledge_base)
    sdt = SemanticDigitalTwin(belief_state)
    warrant_gate = WarrantGate(sdt)
    executive = ExecutionEngine()
    
    print("=== STEP 1: Sensing & Perception Update ===")
    raw_observation = sensing.sense(
        true_ambient_temp=21.5, 
        true_friction=0.45, 
        eva_pos="living_room", 
        busyness="medium"
    )
    perception.update(raw_observation)
    print(f"Raw Sensor Observation : {raw_observation}")
    print(f"Updated Internal Belief : Temp={belief_state.ambient_temperature_c}°C, Friction={belief_state.surface_friction_coefficient}")
    
    print("\n=== STEP 2: Instruction Interpretation ===")
    raw_command = "Bring me my coffee fast!"
    goal = interpreter.interpret(raw_command, belief_state)
    
    print("\n=== STEP 3: System 1 Proposer Generates Parameters ===")
    proposal_v1 = ParameterPacket(
        grasp_pose=(0.45, 0.12, 0.30),
        max_velocity_mps=0.8, 
        tilt_rate_radps=0.2
    )
    print(f"Proposer suggests: v_max = {proposal_v1.max_velocity_mps} m/s")
    
    print("\n=== STEP 4: System 2 SDT & Warrant Gate Validation (Attempt 1) ===")
    warrant_v1 = warrant_gate.validate(proposal_v1, goal, belief_state)
    print(f"SDT Predictions: Nav-Vel = {warrant_v1['predictions']['predicted_nav_velocity']} m/s")
    print(f"Warrant Checks  : {warrant_v1['warrant_checks']}")
    print(f"Gate Verdict    : REJECTED (Failed on: {warrant_v1['failed_criterion']})")
    
    executive.record_trace(raw_command, goal, belief_state, proposal_v1, warrant_v1, raw_observation, committed=False)
    
    print("\n=== STEP 5: Proposer Resamples (Attempt 2) ===")
    proposal_v2 = ParameterPacket(
        grasp_pose=(0.45, 0.12, 0.30),
        max_velocity_mps=0.45, 
        tilt_rate_radps=0.08
    )
    
    warrant_v2 = warrant_gate.validate(proposal_v2, goal, belief_state)
    print(f"SDT Predictions: Force = {warrant_v2['predictions']['predicted_force_n']} N, Wave = {warrant_v2['predictions']['predicted_wave_mm']} mm")
    print(f"Warrant Checks  : {warrant_v2['warrant_checks']}")
    print(f"Gate Verdict    : APPROVED")
    
    print("\n=== STEP 6: Physical Execution & Epistemic Audit Trail ===")
    if warrant_v2["passed"]:
        final_outcome = executive.execute_action(proposal_v2, goal, warrant_v2)
        executive.record_trace(raw_command, goal, belief_state, proposal_v2, warrant_v2, raw_observation, committed=True, execution_outcome=final_outcome)
        print(f"Execution Outcome: {final_outcome}")

    print(f"\n--- COMPLETE EPISODIC AUDIT TRAIL ({len(executive.audit_trail)} Entries) ---")
    for idx, trace in enumerate(executive.audit_trail, 1):
        status = trace['committed']['action_approved']
        outcome = trace.get('execution_outcome')
        outcome_str = f" | Outcome: {outcome['status']} (Spill: {outcome['spill_occurred']})" if outcome else ""
        print(f"Entry #{idx} | Approved: {status}{outcome_str} | Command: '{trace['asked']['raw_utterance']}'")

=== STEP 1: Sensing & Perception Update ===
Raw Sensor Observation : Observation(timestamp=1793013007.6063476, measured_temperature_c=21.08, measured_friction=0.46, eva_position='living_room', room_busyness='medium')
Updated Internal Belief : Temp=21.08°C, Friction=0.46

=== STEP 2: Instruction Interpretation ===

[INTERPRETATION] Analyzing instruction: 'Bring me my coffee fast!' in context of Eva (Position: living_room, Room Busyness: medium)
 -> Derived Goal: Object=blue_ceramic_mug, MaxForce=4.5N, MaxNavVel=0.72m/s, TimeFrame=30.0s

=== STEP 3: System 1 Proposer Generates Parameters ===
Proposer suggests: v_max = 0.8 m/s

=== STEP 4: System 2 SDT & Warrant Gate Validation (Attempt 1) ===
SDT Predictions: Nav-Vel = 0.72 m/s
Warrant Checks  : {'grounded': True, 'feasible': True, 'safe': False, 'monitorable': True, 'repairable': True, 'recordable': True, 'energetically_viable': False}
Gate Verdict    : REJECTED (Failed on: safe)

=== STEP 5: Proposer Resamples (Attempt 2) ===
SDT Predi

## Questions (0.25 Pt x 4 + 0.5 Pt)

### Task 1: Architectural Mapping & Code Decomposition
* **Question 1.1:** Map the classes and methods implemented in your Python script directly onto the functional blocks of the lecture integration diagram. Explain how data flows between these components during a single cognitive decision cycle.

We'll use bullet points to explain each class and method implemented in the Python code:
    - The SensingSystem class collects observations from the environment so as to simulate the robot's sensors;
    - The PerceptionSystem class uses these observations to update the robot's BeliefState, which stores all the information the robot currently know about its environment;
    - The InterpretationSystem class converts the user's natural language instructions (which might be very vague) into specific task instructions/requirements that can be used by the robot, using its BeliefState. It produces a GoalSpecification object that contains the goal (specific instructions usable by the robot) and the constraints (e.g., performing the task quickly);
    - The Proposer class generates possible actions based on the task requirements (they must be checked by the WarrantGate before being executed);
    - The SemanticDigitalTwin class uses a model of the environment to predict the consequences of an action, before the robot physically executes it;
    - The WarrantGate class compares the predictions from the SemanticDigitalTwin to the task requirement. It then evaluates whether that action can be approved (be justifiable) or not;
    - The ExecutingEngine class executes an approved action (execute_action() method) and records the decisions, observations and outcomes in an output_trail list (record_trace() method);


Let's know explain how the whole cycle works. The robot begins by collecting information about its environment using SensingSystem. The PerceptionSystem class then uses these observation to update the robot's belief state, which stores all the information the robot currently knows about its environment. Then, the InterpretationSystem converts the user's natural language request into a GoalSpecification object that contains the task requirements and constraints.
The Proposer class then generates possible actions, which are stored as ParameterPacket objects. The SemanticDigitalTwin class predicts the consequences of executing those actions, and the WarrantGate class verifies whether executing those actions respects the task requirements and constraints.
Finally, if a proposed action passes all the tests, the ExecutionEngine executes that action and stores some information about the decisions, observations and outcomes in the audit trail list. Information about rejected actions is also stored.


    
* **Question 1.2:** Complete the proposer class based on a probabilistic model of your choice (e.g., uniform, gaussian, mixture). Justify your choice.

I decided to use a uniform distribution to generate the maximum speed and tilt rate values because it is simple to understand and implement. It also allows me to easily modify the interval ranges to explore different possible actions. Since every value in an interval range has the same odds of being selected, the Proposer class can generate a variety of actions that will later be predicted by the SemanticDigitalTwin and evaluated by the WarrantGate.
  

### Task 2: Non-Linear Contact Mechanics and Material Compliance
In the base script, the contact force on the flexible pouch is modeled using a simplified power-law equation. 
* **Question 2.1:** Modify the `SemanticDigitalTwin.predict_properties` method to account for a dual-stage material compliance curve. If the contact depth exceeds $0.02 \, \text{m}$, the stiffness coefficient should increase exponentially by a factor of $1.8$ due to bottoming out against a rigid internal fixture.

Here, we are told that if a robot presses against a pouch but does not go deeper than 2cm, the pouch has normal stiffness. However, if it goes over that threshold, it should increase exponentially (1.8 to the power of the excess depth over 2m). We suppose it increases exponentially every meter that goes over the 2 meter depth.


* **Question 2.2:** Write a test case where the Proposer generates an aggressive approach velocity ($0.35 \, \text{m/s}$). Calculate the predicted force and verify that the Warrant Gate successfully intercepts and rejects the parameter packet before physical execution.

We have tested this question in the last cell of this notebook. The WarrantGate rejects the proposed action because the predicted contact force of 4.76 N (example of a value for intensity of the predicted force) exceeds the maximum allowed force of 4.5 N (the safety requirement is not satisfied). Since the WarrantGate rejects the action, result["passed"] is False and the execute_action() method is not called (the action is not executed).

### Task 3: Advanced Fluid Sloshing and Viscosity Temperature Coupling
Fluids in laboratory automation often exhibit temperature-dependent viscosity changes.
* **Question 3.1:** Enhance the fluid dynamics model so that ambient temperature fluctuations directly alter the effective fluid viscosity using an Arrhenius-type approximation or a linear thermal degradation coefficient ($1\% \, \text{drop per} \, ^\circ\text{C}$ above $20^\circ\text{C}$).
* **Question 3.2:** Simulate an environment where the ambient temperature rises to $28^\circ\text{C}$. How does this change the predicted wave height for a tilt rate of $0.15 \, \text{rad/s}$? Does the Warrant Gate trigger a safety violation?

  Increasing the temperature form 20 degrees Celsius to 28 degrees Celsius decreases the fluid's viscosity, causing the liquid to move more inside the mug. Consequently, its predicted wave height will increase, making it less safe for the robot to move with the mug because there is a higher risk of the coffee spilling over.

### Task 4: Expanding the Warrant Gate Checklist
The current Warrant Gate evaluates six core criteria (*grounded, feasible, safe, monitorable, repairable, recordable*). 
* **Question 4.1:** Add a seventh criterion called **`energetically_viable`** to the Warrant Gate dictionary checks. It must evaluate whether the estimated battery drain for the requested trajectory stays below $15\%$ of total capacity.
* **Question 4.2:** Introduce a low-battery state into the `BeliefState` and demonstrate how a failed `energetically_viable` check redirects the system workflow to request a recharge action rather than proceeding with the coffee delivery.

  Let's suppose,like in the tests below, that the robot currently has its battery at 5% and is asked to perform a task that requires 10%. It will therefore not execute that action, even though that action's battery drain is lower than the 15% threshold. Indeed, the Warrant Gate rejects that action and the robot requests a recharge.

### Task 5: Post-Hoc Epistemic Audit Analysis
A core pillar of trustworthy robotics is the ability to audit *why* a decision was made or rejected.
* **Question 5.1:** Write a post-processing function (`audit_analyzer(audit_trail)`) that parses the recorded episodic log. It should compute and print:
  1. The total number of rejected proposals versus approved executions.
  2. The most frequent warrant criterion responsible for rejection.
  3. The average variance between predicted forces and observed physical outcomes.
* **Question 5.2:** Discuss in your lab report: *Why does maintaining separate records for `asked`, `belief`, `committed`, and `observed` solve the epistemological gap that pure data-driven imitation learning cannot address?*

## Part 3: Robot Anatomy (0.5 Pt)

**Title:** Links Vs Joints, Actuators Vs Sensors, Effectors Vs Bases Vs Torsos Vs Heads, Wheels Vs Legs Vs Grippers, Proprioception Vs Exteroception Vs Interoception

![Alternative Text](robot-body.jpg)

### Questions

Identify the key functional parts of a robot based on the above picture and briefly indicate how they support the execution of function in the above architecture.

    Let's go through the terms in order.
    -The joints allow the robot's arms move to move, while the links are the solid parts of its arms;
    - The actuators are the "muscles" of the robot as they are motors that move the joints and wheels;
    - The sensors are cameras and laser scanners that feed information on the environment to the robot;
    - The effectors are any devices attached at the end of a robot's arm, and that allow it to interact with the environment;
    - The robot's base is the lower structure that allows it to move;
    - The robot's torso supports its head and arms, while the head contains cameras the robot uses to perceive the environment;
    - Both the wheels and the legs allow the robot to move through the environment. In this case the robot has wheels;
    - The grippers allow the robot to grab objects and are a type of effector;
    - Proprioception is the robot's ability to sense its own movement and orientation in space;
    - Exteroception is the process through which the robot detects and processes information origination from outside its body, usually through its cameras;
    - Finally, interoception is the system that allows the robot to detect and interpret signals from inside its body, e.g., battery level or the temperature of its motors

---

# Submission guidelines

- Deadline: 03.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (Create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put all the answers in the notebook. Any other media related to your answer should be inserted in the notebook as well. The notebook and all related media should then be put into a folder called **Assignment1**. 

In [94]:
#Tests

#Part 1
proposer = Proposer()
action1 = proposer.propose()
action2 = proposer.propose()
print(action1)
print(action2)


#Part 2.1
#Normal stiffness contact depth <0.02
proposal1 = ParameterPacket(grasp_pose=(0.45, 0.12, 0.30), max_velocity_mps=0.35, tilt_rate_radps=0.01, contact_depth_m=0.015)

#Increased stiffness 
proposal2 = ParameterPacket(grasp_pose=(0.45, 0.12, 0.30), max_velocity_mps=0.35, tilt_rate_radps=0.01, contact_depth_m=0.03)
force1 = sdt.predict_properties(proposal1, goal)["predicted_force_n"]
force2 = sdt.predict_properties(proposal2, goal)["predicted_force_n"]

print("\nPart 2.1")
print("Normal stiffness force:", force1)
print("Increased stiffness force:", force2)

#Part 2.2
#Test if WarrantGate rejects the ParameterPacket if the speed is set at 0.35
#We can increase the contact depth with a speed of 0.35 until the predicted force goes over 4.5 (max force for the blue mug) -> proposal2

print("\nPart 2.2")

#However we can also specify the speed specifically as 0.35
agr_action = proposer.propose(speed=0.35)
agr_action.contact_depth_m = 0.03
agr_action.tilt_rate_radps = 0.01

#Run the warrant gate to verify if the action is justifiable
result = warrant_gate.validate(agr_action, goal, belief_state)

print("Predicted force", result['predictions']['predicted_force_n'])

#Is the action executed, i.e., passes the warrant gate and goes to the InteractionEngine
if result["passed"]:
    executive.execute_action(action2, goal, result)
else:
    print("Action rejected")

print("Failed criterion:", result["failed_criterion"])


#Part 3
#We will set the temp in the robot's belief state to 28 deg -> 8 deg over 20 so viscosity decreases by 8percent; liquid mvmnt speed should increase (inversely proportional to viscosity)
belief_state.ambient_temperature_c = 28.0

#tilt rate set to 0.15 in both temperature cases
test_action = ParameterPacket(grasp_pose=(0.45, 0.12, 0.30), max_velocity_mps=0.2, tilt_rate_radps=0.15)
prediction = sdt.predict_properties(test_action, goal)
print("\nPart 3\nTemperature=28 degrees\nPredicted wave height:", prediction["predicted_wave_mm"])

#Does the predicted wave height go over the threshold for wave height ->wave height
result = warrant_gate.validate(test_action, goal, belief_state)

print("Approved:", result["passed"])
print("Failed criterion:", result["failed_criterion"])

#Now let's test at 20 degrees
belief_state.ambient_temperature_c = 20.0
prediction = sdt.predict_properties(test_action, goal)
print("\nTemperature=20 degrees\nTemperature:", belief_state.ambient_temperature_c)
print("Predicted wave height:", prediction["predicted_wave_mm"])

result = warrant_gate.validate(test_action, goal, belief_state)

print("Approved:", result["passed"])
print("Failed criterion:", result["failed_criterion"])


#Part 4.1
print("\nPart 4.1\nAction 1 (battery check fails)")
#We are going to test the energetically viable parameter -> should be lower than 15

#if speed= 0.5 -> 0.5 * 40 = 20 > 15 ======= warrant gate battery check fails
action_bat = proposer.propose(speed=0.5)

result = warrant_gate.validate(action_bat, goal, belief_state)

print("Estimated battery drain:", action_bat.max_velocity_mps * 40.0, "%")
print("Energetically viable:", result["warrant_checks"]["energetically_viable"])

#if speed= 0.25 -> 0.25 *40=10 < 15 ===== warrant gate battery check passes
action_bat_suc = proposer.propose(speed=0.25)
result = warrant_gate.validate(action_bat_suc, goal, belief_state)

print("\nAction 2 (battery check passes)\nEstimated battery drain:", action_bat_suc.max_velocity_mps * 40.0, "%")
print("Energetically viable:", result["warrant_checks"]["energetically_viable"])

#Part 4.2
#We will simulate a robot with low battery 5%
print("\nPart 4.2")

low_battery_belief = BeliefState(battery_level_percent=5.0)

#print("Battery level:", low_battery_belief.battery_level_percent, "%")

#lets take the action calculate its battery drain, and then see if the current battery is enough to execute that action
#required_battery = action_bat_suc.max_velocity_mps * 40.0


#if low_battery_belief.battery_level_percent < required_battery:
#    print("Not enough battery")
#else:
#    print("Enough battery")

result = warrant_gate.validate(action_bat_suc, goal, low_battery_belief)

print("Battery remaining:", low_battery_belief.battery_level_percent, "%")
print("Battery required:", action_bat_suc.max_velocity_mps * 40.0, "%")
print("Energetically viable:", result["warrant_checks"]["energetically_viable"])

required_battery = action_bat_suc.max_velocity_mps * 40.0

if low_battery_belief.battery_level_percent < required_battery:
    print("Recharge requested")
elif result["passed"]:
    executive.execute_action(action_bat_suc, goal, result)
else:
    print("Action rejected by the Warrant Gate")

ParameterPacket(grasp_pose=(0.45, 0.12, 0.3), max_velocity_mps=0.4620101116735992, tilt_rate_radps=0.06726338516655213, source_generator='stochastic_sampler', contact_depth_m=0.0)
ParameterPacket(grasp_pose=(0.45, 0.12, 0.3), max_velocity_mps=0.2930157867799674, tilt_rate_radps=0.030657131065602278, source_generator='stochastic_sampler', contact_depth_m=0.0)

Part 2.1
Normal stiffness force: 2.99
Increased stiffness force: 4.8

Part 2.2
Predicted force 4.78
Action rejected
Failed criterion: safe

Part 3
Temperature=28 degrees
Predicted wave height: 6.97
Approved: False
Failed criterion: safe

Temperature=20 degrees
Temperature: 20.0
Predicted wave height: 6.25
Approved: False
Failed criterion: safe

Part 4.1
Action 1 (battery check fails)
Estimated battery drain: 20.0 %
Energetically viable: False

Action 2 (battery check passes)
Estimated battery drain: 10.0 %
Energetically viable: True

Part 4.2
Battery remaining: 5.0 %
Battery required: 10.0 %
Energetically viable: False
Recharge re